# 6조 타이타닉 모델 개선 · API · 웹페이지

이 노트북은 원본 6.csv를 사용하여 모델을 다시 학습하고 웹페이지를 실행합니다.
사전 로컬 실행 결과는 **74.37%**로 **목표 85% 미달성**입니다. 출력값을 조작하지 않습니다.
노트북 자체의 실행 출력은 비워두었습니다. 위에서부터 실행하면 현재 실행 결과가 표시됩니다.

## 1. 프로젝트 폴더와 패키지 준비
패키지 설치는 별도 프로세스에서 실행합니다. 기존 원본 노트북은 수정하지 않습니다.

In [ ]:
from pathlib import Path
import subprocess, sys, os, json
project=Path('/content/titanic_team6')
project.mkdir(exist_ok=True)
os.chdir(project)


In [ ]:
requirements='numpy==2.3.5\npandas==2.2.3\njoblib==1.5.3\nscipy==1.17.0\nthreadpoolctl==3.6.0\nscikit-learn==1.8.0\n'
Path('requirements.txt').write_text(requirements)
subprocess.run([sys.executable,'-m','pip','install','-q','-r','requirements.txt'],check=True)


## 2. 학습·API·웹페이지 코드 생성
아래 셀은 모두 실행해야 합니다. 각 문자열은 대응 파일의 전체 소스입니다.

In [ ]:
# train.py
p=Path('train.py')
p.parent.mkdir(parents=True,exist_ok=True)
p.write_text('"""Reproducible training: validation selects model, held-out test evaluates once."""\nimport os\nos.environ.setdefault(\'OMP_NUM_THREADS\',\'4\')\nimport argparse, hashlib, json, platform\nfrom pathlib import Path\nimport joblib, numpy as np, pandas as pd, sklearn\nfrom sklearn.compose import ColumnTransformer\nfrom sklearn.pipeline import make_pipeline\nfrom sklearn.preprocessing import OneHotEncoder, OrdinalEncoder, StandardScaler\nfrom sklearn.impute import SimpleImputer\nfrom sklearn.ensemble import HistGradientBoostingClassifier, RandomForestClassifier\nfrom sklearn.linear_model import LogisticRegression\nfrom sklearn.model_selection import train_test_split\nfrom sklearn.metrics import accuracy_score, balanced_accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, confusion_matrix\nfrom sklearn.base import clone\n\nROOT=Path(__file__).resolve().parent\nEXCLUDE=[\'full_name\',\'name\',\'survived\',\'survival_probability\',\'family_survival_rate\']\ndef load_data(path):\n    # Literal "None" means no occupation/medical condition, not a missing value.\n    df=pd.read_csv(path,keep_default_na=False,na_values=[\'\'])\n    if \'survived\' not in df or df.survived.isna().any() or not set(df.survived.unique()) <= {0,1}:\n        raise ValueError(\'survived must contain non-missing 0/1 labels\')\n    X=df.drop(columns=[c for c in df if c in EXCLUDE or c.startswith(\'Unnamed:\')])\n    return df,X,df.survived.astype(int)\n\ndef metrics(y,p):\n    pred=(p>=.5).astype(int)\n    acc=float(accuracy_score(y,pred));n=len(y);z=1.96\n    center=(acc+z*z/(2*n))/(1+z*z/n)\n    half=z*np.sqrt(acc*(1-acc)/n+z*z/(4*n*n))/(1+z*z/n)\n    return dict(accuracy=acc,accuracy_ci95=[float(center-half),float(center+half)],balanced_accuracy=float(balanced_accuracy_score(y,pred)),precision=float(precision_score(y,pred,zero_division=0)),recall=float(recall_score(y,pred,zero_division=0)),f1=float(f1_score(y,pred,zero_division=0)),roc_auc=float(roc_auc_score(y,p)),confusion_matrix=confusion_matrix(y,pred,labels=[0,1]).tolist())\n\ndef main(path):\n    out=ROOT/\'artifacts\';out.mkdir(exist_ok=True)\n    df,X,y=load_data(path)\n    # Preserve original notebook\'s exact held-out 20% split for comparability.\n    Xdev,Xtest,ydev,ytest=train_test_split(X,y,test_size=.2,random_state=42)\n    Xt,Xv,yt,yv=train_test_split(Xdev,ydev,test_size=.25,random_state=43,stratify=ydev)\n    cats=X.select_dtypes(include=[\'object\']).columns.tolist();nums=[c for c in X if c not in cats]\n    catpipe=make_pipeline(SimpleImputer(strategy=\'constant\',fill_value=\'__missing__\'),OrdinalEncoder(handle_unknown=\'use_encoded_value\',unknown_value=-1))\n    ordinal=ColumnTransformer([(\'numeric\',SimpleImputer(strategy=\'median\'),nums),(\'category\',catpipe,cats)])\n    onehot=ColumnTransformer([(\'numeric\',make_pipeline(SimpleImputer(strategy=\'median\'),StandardScaler()),nums),(\'category\',make_pipeline(SimpleImputer(strategy=\'constant\',fill_value=\'__missing__\'),OneHotEncoder(handle_unknown=\'ignore\')),cats)])\n    mask=[False]*len(nums)+[True]*len(cats)\n    candidates={\n      \'random_forest_depth5\':make_pipeline(clone(ordinal),RandomForestClassifier(n_estimators=100,max_depth=5,random_state=42,n_jobs=4)),\n      \'logistic_regression\':make_pipeline(onehot,LogisticRegression(C=1,max_iter=1000)),\n      \'hist_gradient_7leaves\':make_pipeline(clone(ordinal),HistGradientBoostingClassifier(max_iter=300,max_leaf_nodes=7,learning_rate=.05,l2_regularization=10,categorical_features=mask,random_state=42)),\n      \'hist_gradient_15leaves\':make_pipeline(clone(ordinal),HistGradientBoostingClassifier(max_iter=300,max_leaf_nodes=15,learning_rate=.05,l2_regularization=10,categorical_features=mask,random_state=42)),\n      \'hist_gradient_31leaves\':make_pipeline(clone(ordinal),HistGradientBoostingClassifier(max_iter=200,max_leaf_nodes=31,learning_rate=.05,l2_regularization=10,categorical_features=mask,random_state=42))}\n    results=[]\n    for name,model in candidates.items():\n        model.fit(Xt,yt);score=metrics(yv,model.predict_proba(Xv)[:,1]);results.append(dict(model=name,**score));print(name,score[\'accuracy\'],flush=True)\n    winner=max(results,key=lambda r:r[\'accuracy\'])[\'model\']\n    final=clone(candidates[winner]);final.fit(Xdev,ydev)\n    test=metrics(ytest,final.predict_proba(Xtest)[:,1])\n    baseline=clone(candidates[\'random_forest_depth5\']);baseline.fit(Xdev,ydev)\n    base_metrics=metrics(ytest,baseline.predict_proba(Xtest)[:,1])\n    p=df.survival_probability.to_numpy() if \'survival_probability\' in df else None\n    diagnostic={} if p is None else {\'probability_threshold_accuracy_all_rows\':float(np.mean((p>=.5)==y)), \'expected_accuracy_if_labels_are_bernoulli\':float(np.mean(np.maximum(p,1-p))), \'interpretation\':\'Conditional diagnostic, NOT a proven ceiling. Only applies if supplied probability is the true P(y=1|X). Not used for training or model selection.\'}\n    report={\'rows\':len(df),\'features\':X.columns.tolist(),\'excluded\':[c for c in df if c not in X], \'split\':{\'train\':len(Xt),\'validation\':len(Xv),\'test\':len(Xtest),\'test_seed\':42,\'validation_seed\':43,\'final_fit_rows\':len(Xdev)},\'selection\':\'Validation accuracy; threshold fixed at 0.5. Final winner refit on train+validation; test not used to select.\',\'validation_candidates\':results,\'selected_model\':winner,\'test\':test,\'corrected_baseline_test\':base_metrics,\'original_notebook_saved_accuracy\':.7353,\'original_notebook_saved_improved_accuracy\':.7430,\'target_accuracy\':.85,\'target_met\':test[\'accuracy\']>=.85,\'probability_diagnostic\':diagnostic,\'csv_sha256\':hashlib.sha256(Path(path).read_bytes()).hexdigest(),\'versions\':{\'python\':platform.python_version(),\'sklearn\':sklearn.__version__,\'pandas\':pd.__version__,\'numpy\':np.__version__,\'joblib\':joblib.__version__},\'missingness\':{\'development\':Xdev.isna().sum().to_dict(),\'test\':Xtest.isna().sum().to_dict()},\'scope\':\'Educational evacuation scenario: lifeboat_access, emergency_response_time, evacuation_priority must be available at prediction time. Not a pre-boarding survival forecast; provenance of family_risk_score must be confirmed.\'}\n    fields=[]\n    example_row=Xdev.dropna().iloc[0]\n    for c in X:\n        s=Xdev[c]\n        if c in cats:\n            choices=sorted(s.dropna().unique().tolist())\n            fields.append({\'name\':c,\'type\':\'category\',\'choices\':choices,\'default\':str(example_row[c]),\'nullable\':bool(X[c].isna().any())})\n        else:\n            integer=pd.api.types.is_integer_dtype(s.dtype)\n            fields.append({\'name\':c,\'type\':\'integer\' if integer else \'number\',\'min\':float(s.min()),\'max\':float(s.max()),\'default\':int(example_row[c]) if integer else float(example_row[c]),\'nullable\':bool(X[c].isna().any())})\n    schema={\'fields\':fields,\'output\':{\'survived\':\'0 or 1\',\'survival_probability\':\'model-estimated probability between 0 and 1\'},\'threshold\':.5,\'model\':winner,\'test_accuracy\':test[\'accuracy\'],\'target_met\':report[\'target_met\'],\'scenario\':\'대피 상황 정보가 주어진 교육용 생존 예측\'}\n    joblib.dump(final,out/\'model.joblib\',compress=3)\n    for name,obj in [(\'metrics.json\',report),(\'schema.json\',schema)]: (out/name).write_text(json.dumps(obj,ensure_ascii=False,indent=2),encoding=\'utf-8\')\n    example={f[\'name\']:f[\'default\'] for f in fields}\n    (ROOT/\'example_request.json\').write_text(json.dumps(example,ensure_ascii=False,indent=2),encoding=\'utf-8\')\n    print(json.dumps({\'selected\':winner,\'test\':test,\'target_met\':report[\'target_met\']},indent=2),flush=True)\nif __name__==\'__main__\':\n    parser=argparse.ArgumentParser();parser.add_argument(\'--data\',default=str(ROOT/\'data\'/\'6.csv\'));args=parser.parse_args();main(args.data)\n',encoding='utf-8')
print('생성:',p)


In [ ]:
# app.py
p=Path('app.py')
p.parent.mkdir(parents=True,exist_ok=True)
p.write_text('"""Local educational JSON API + same-origin web UI (Python standard library)."""\nimport os\nos.environ.setdefault(\'OMP_NUM_THREADS\',\'4\')\nimport argparse, json, math, threading\nfrom http.server import BaseHTTPRequestHandler, ThreadingHTTPServer\nfrom pathlib import Path\nfrom urllib.parse import urlsplit\nimport joblib, pandas as pd\nROOT=Path(__file__).resolve().parent\nclass Predictor:\n    def __init__(self):\n        self.model=joblib.load(ROOT/\'artifacts/model.joblib\')\n        self.schema=json.loads((ROOT/\'artifacts/schema.json\').read_text(encoding=\'utf-8\'))\n        self.lock=threading.Lock()\n    def predict(self,payload):\n        if not isinstance(payload,dict): raise ValueError(\'JSON 객체를 보내주세요.\')\n        fields=self.schema[\'fields\']; names={f[\'name\'] for f in fields}\n        extra=set(payload)-names; missing=names-set(payload)\n        if extra: raise ValueError(\'허용되지 않은 항목: \'+\', \'.join(sorted(extra)))\n        if missing: raise ValueError(\'누락된 항목: \'+\', \'.join(sorted(missing)))\n        row={}\n        for f in fields:\n            k=f[\'name\'];v=payload[k]\n            if v is None:\n                if not f[\'nullable\']: raise ValueError(k+\': 필수 입력입니다.\')\n                row[k]=float(\'nan\');continue\n            if f[\'type\']==\'category\':\n                if not isinstance(v,str) or v not in f[\'choices\']: raise ValueError(k+\': 허용된 선택지를 사용하세요.\')\n            else:\n                if isinstance(v,bool) or not isinstance(v,(int,float)) or not math.isfinite(v): raise ValueError(k+\': 유한한 숫자를 입력하세요.\')\n                if f[\'type\']==\'integer\' and int(v)!=v: raise ValueError(k+\': 정수를 입력하세요.\')\n                if not f[\'min\']<=v<=f[\'max\']: raise ValueError(f"{k}: 학습 범위 {f[\'min\']} ~ {f[\'max\']} 안에서 입력하세요.")\n            row[k]=v\n        frame=pd.DataFrame([row],columns=[f[\'name\'] for f in fields])\n        for f in fields:\n            if f[\'type\']==\'category\': frame[f[\'name\']]=frame[f[\'name\']].astype(object)\n        with self.lock:\n            p=float(self.model.predict_proba(frame)[0,1])\n        survived=int(p>=self.schema[\'threshold\'])\n        return {\'survived\':survived,\'label\':\'생존 예측\' if survived else \'비생존 예측\',\'survival_probability\':p,\'threshold\':self.schema[\'threshold\'],\'model\':self.schema[\'model\']}\n\ndef make_server(host=\'127.0.0.1\',port=8000):\n    predictor=Predictor()\n    class Handler(BaseHTTPRequestHandler):\n        def send(self,status,body,mime=\'application/json; charset=utf-8\'):\n            if not isinstance(body,bytes): body=json.dumps(body,ensure_ascii=False,allow_nan=False).encode()\n            self.send_response(status);self.send_header(\'Content-Type\',mime);self.send_header(\'Content-Length\',str(len(body)));self.send_header(\'X-Content-Type-Options\',\'nosniff\');self.send_header(\'Cache-Control\',\'no-store\');self.end_headers();self.wfile.write(body)\n        def do_GET(self):\n            path=urlsplit(self.path).path.rstrip(\'/\') or \'/\'\n            if path==\'/\': self.send(200,(ROOT/\'static/index.html\').read_bytes(),\'text/html; charset=utf-8\')\n            elif path==\'/api/schema\': self.send(200,predictor.schema)\n            elif path==\'/api/metrics\': self.send(200,json.loads((ROOT/\'artifacts/metrics.json\').read_text(encoding=\'utf-8\')))\n            elif path==\'/api/health\': self.send(200,{\'status\':\'ok\',\'model_loaded\':True})\n            else: self.send(404,{\'error\':\'경로를 찾을 수 없습니다.\'})\n        def do_POST(self):\n            if urlsplit(self.path).path.rstrip(\'/\')!=\'/api/predict\': return self.send(404,{\'error\':\'경로를 찾을 수 없습니다.\'})\n            if self.headers.get(\'Content-Type\',\'\').split(\';\')[0].strip()!=\'application/json\': return self.send(415,{\'error\':\'Content-Type은 application/json이어야 합니다.\'})\n            try: size=int(self.headers.get(\'Content-Length\',\'0\'))\n            except ValueError: return self.send(400,{\'error\':\'잘못된 Content-Length\'})\n            if not 0<size<=32768: return self.send(413,{\'error\':\'요청 크기는 1~32768바이트여야 합니다.\'})\n            try:\n                self.connection.settimeout(10)\n                payload=json.loads(self.rfile.read(size).decode(\'utf-8\'))\n                result=predictor.predict(payload)\n            except (ValueError,UnicodeError,TimeoutError) as e: return self.send(422,{\'error\':str(e)})\n            except Exception:\n                self.log_error(\'prediction failed\');return self.send(500,{\'error\':\'예측 처리 중 오류가 발생했습니다.\'})\n            self.send(200,result)\n    return ThreadingHTTPServer((host,port),Handler)\nif __name__==\'__main__\':\n    parser=argparse.ArgumentParser();parser.add_argument(\'--host\',default=\'127.0.0.1\');parser.add_argument(\'--port\',type=int,default=8000);args=parser.parse_args()\n    try: server=make_server(args.host,args.port)\n    except FileNotFoundError: raise SystemExit(\'학습 파일이 없습니다. 먼저 python train.py --data data/6.csv 를 실행하세요.\')\n    print(f\'Open http://{args.host}:{args.port}\',flush=True)\n    try: server.serve_forever()\n    except KeyboardInterrupt: server.server_close()\n',encoding='utf-8')
print('생성:',p)


In [ ]:
# static/index.html
p=Path('static/index.html')
p.parent.mkdir(parents=True,exist_ok=True)
p.write_text('<!doctype html><html lang="ko"><head><meta charset="utf-8"><meta name="viewport" content="width=device-width,initial-scale=1"><title>6조 · 타이타닉 생존 예측</title>\n<style>\n:root{font-family:Arial,\'Malgun Gothic\',sans-serif;color:#eaf1f7;background:#0d1723;color-scheme:dark}*{box-sizing:border-box}body{margin:0}main{max-width:1180px;margin:auto;padding:44px 24px}header{margin-bottom:32px}.eyebrow{color:#69dbbe;letter-spacing:3px;font-size:12px;font-weight:700}h1{font-size:36px;letter-spacing:-1.5px;margin:14px 0}p{color:#a8b9c8;line-height:1.7}.layout{display:grid;grid-template-columns:1fr 340px;gap:24px}.card{background:#152333;border:1px solid #2a3b4c;border-radius:18px;padding:26px}aside{align-self:start;position:sticky;top:20px}h2{font-size:19px;margin-top:0}.fields{display:grid;grid-template-columns:1fr 1fr;gap:20px}label{font-size:13px;color:#d8e5ee;display:block}label small{display:block;color:#90a6b8;margin:5px 0 8px;font-size:11px}input,select{width:100%;padding:12px;border:1px solid #405367;border-radius:8px;background:#0e1a28;color:#fff;font:inherit}input:focus,select:focus{outline:2px solid #69dbbe}button{cursor:pointer;border:0;background:#69dbbe;color:#092019;padding:15px 22px;border-radius:9px;font-size:15px;font-weight:bold}button:disabled{opacity:.5;cursor:wait}.actions{display:flex;gap:12px;margin-top:26px;flex-wrap:wrap}.secondary{background:#293d50;color:#dce9f2}.stat{font-size:42px;color:#69dbbe;font-weight:bold}.muted{font-size:13px}.divider{border-top:1px solid #324456;margin:24px 0}#error{color:#ffb6aa;white-space:pre-wrap}progress{width:100%;height:12px;accent-color:#69dbbe}pre{white-space:pre-wrap;overflow-wrap:anywhere;background:#0e1a28;padding:14px;font-size:12px;border-radius:8px}details{margin-top:22px}summary{cursor:pointer;color:#a7bdce}#status{color:#ffd596;font-size:13px}.hidden{display:none}@media(max-width:850px){.layout{grid-template-columns:1fr}aside{position:static;grid-row:1}h1{font-size:29px}}@media(max-width:480px){.fields{grid-template-columns:1fr}main{padding:28px 16px}.card{padding:20px}}\n</style></head><body><main><header><div class="eyebrow">TEAM 06 / MACHINE LEARNING</div><h1>타이타닉 생존 예측</h1><p>승객 및 대피 상황 정보를 입력하면 학습된 모델이 생존 여부와 확률을 예측합니다.<br>대피 시점 정보가 주어진 교육용 시뮬레이션입니다.</p></header><div class="layout"><section class="card"><h2>예측에 사용할 정보</h2><p class="muted">기본 예시를 불러온 뒤 값을 변경해보세요. 숫자는 학습 데이터 범위 안에서 입력합니다.<br>‘정보 없음’은 결측값, ‘None’은 해당 사항 없음을 뜻합니다.</p><form id="form"><div id="fields" class="fields"></div><div class="actions"><button id="submit" disabled>생존 여부 예측하기</button><button class="secondary" type="button" id="reset">기본 예시 복원</button></div></form><p id="error" role="alert"></p><details><summary>실제 API 요청과 응답 보기</summary><p class="muted">POST api/predict · application/json</p><pre id="request">예측하면 요청 JSON이 표시됩니다.</pre><pre id="response">응답 대기 중</pre></details></section><aside class="card"><h2>모델 평가</h2><div id="accuracy" class="stat">—</div><p class="muted">학습에 사용하지 않은 테스트 데이터 정확도</p><div id="status">모델 정보를 불러오는 중…</div><div class="divider"></div><h2>예측 결과</h2><div id="result" aria-live="polite"><p>정보를 입력하고 예측 버튼을 눌러주세요.</p></div><p class="muted">개별 생존 확률과 모델 전체 정확도는 서로 다른 값입니다.</p><details><summary>평가 방식</summary><p class="muted">훈련 60% · 검증 20% · 테스트 20%. 검증 정확도로 모델을 선택하고, 훈련+검증 80%로 재학습했습니다. 분류 기준은 생존 확률 50%입니다.</p></details></aside></div></main><script>\nconst $=id=>document.getElementById(id);\nconst labels={age:\'나이\',gender:\'성별\',marital_status:\'혼인 상태\',occupation:\'직업\',class:\'객실 등급\',ticket_price:\'티켓 가격\',cabin_deck:\'객실 갑판\',embarkation_port:\'탑승 항구\',travel_purpose:\'여행 목적\',journey_duration_days:\'여행 기간 (일)\',siblings_spouses:\'형제자매·배우자 수\',parents_children:\'부모·자녀 수\',family_size:\'전체 가족 수\',is_alone:\'혼자 탑승 (1=예, 0=아니요)\',boarding_position:\'탑승 순서\',lifeboat_access:\'구명정 접근도\',medical_condition:\'건강 상태\',mobility_score:\'이동 능력 점수\',emergency_response_time:\'비상 대응 시간\',evacuation_priority:\'대피 우선순위\',age_group:\'연령 그룹\',fare_per_person:\'1인당 요금\',family_risk_score:\'가족 위험 점수\'};\nlet schema;\nasync function api(path,options){const r=await fetch(new URL(path,location.href),options);const data=await r.json();if(!r.ok)throw new Error(data.error||\'API 요청 실패\');return data;}\nfunction defaults(){if(!schema)return;for(const f of schema.fields)$(\'f_\'+f.name).value=String(f.default);$(\'error\').textContent=\'\';}\nasync function init(){try{schema=await api(\'api/schema\');for(const f of schema.fields){const label=document.createElement(\'label\');label.htmlFor=\'f_\'+f.name;label.append(document.createTextNode(labels[f.name]||f.name));const hint=document.createElement(\'small\');hint.textContent=f.name+(f.type===\'category\'?\'\':` · ${f.min} ~ ${f.max}`);label.append(hint);const el=document.createElement(f.type===\'category\'?\'select\':\'input\');el.id=\'f_\'+f.name;if(f.type===\'category\'){for(const choice of f.choices){const o=document.createElement(\'option\');o.value=choice;o.textContent=choice;el.append(o);}if(f.nullable){const o=document.createElement(\'option\');o.value=\'\';o.textContent=\'정보 없음\';el.append(o);}}else{el.type=\'number\';el.step=f.type===\'integer\'?\'1\':\'any\';el.min=f.min;el.max=f.max;el.required=!f.nullable;}label.append(el);$(\'fields\').append(label);}defaults();$(\'accuracy\').textContent=(schema.test_accuracy*100).toFixed(2)+\'%\';$(\'status\').textContent=schema.target_met?\'목표 85% 달성\':\'목표 85% 미달성 · 실제 측정 결과\';$(\'submit\').disabled=false;}catch(e){$(\'error\').textContent=\'모델을 불러오지 못했습니다: \'+e.message;$(\'status\').textContent=\'서버 상태를 확인해주세요.\'}}\n$(\'reset\').onclick=defaults;\n$(\'form\').onsubmit=async e=>{e.preventDefault();if(!schema)return;$(\'error\').textContent=\'\';$(\'submit\').disabled=true;$(\'result\').textContent=\'예측 중…\';const body={};for(const f of schema.fields){const v=$(\'f_\'+f.name).value;body[f.name]=v===\'\'?null:f.type===\'category\'?v:Number(v);}$(\'request\').textContent=JSON.stringify(body,null,2);try{const r=await api(\'api/predict\',{method:\'POST\',headers:{\'Content-Type\':\'application/json\'},body:JSON.stringify(body)});$(\'response\').textContent=JSON.stringify(r,null,2);$(\'result\').replaceChildren();const h=document.createElement(\'h2\');h.textContent=r.label;const p=document.createElement(\'p\');p.textContent=\'생존 확률 \'+(r.survival_probability*100).toFixed(1)+\'%\';const bar=document.createElement(\'progress\');bar.max=1;bar.value=r.survival_probability;bar.setAttribute(\'aria-label\',\'생존 확률\');$(\'result\').append(h,p,bar);}catch(err){$(\'error\').textContent=err.message;$(\'result\').textContent=\'예측 실패\';$(\'response\').textContent=err.message;}finally{$(\'submit\').disabled=false}};init();\n</script></body></html>\n',encoding='utf-8')
print('생성:',p)


In [ ]:
# test_api.py
p=Path('test_api.py')
p.parent.mkdir(parents=True,exist_ok=True)
p.write_text("import json, threading, unittest, urllib.request, urllib.error\nimport numpy as np, pandas as pd\nfrom app import ROOT, Predictor, make_server\nclass ApiTests(unittest.TestCase):\n    @classmethod\n    def setUpClass(cls):\n        cls.server=make_server(port=0);threading.Thread(target=cls.server.serve_forever,daemon=True).start()\n        cls.url=f'http://127.0.0.1:{cls.server.server_port}';cls.sample=json.loads((ROOT/'example_request.json').read_text())\n    @classmethod\n    def tearDownClass(cls): cls.server.shutdown();cls.server.server_close()\n    def post(self,data):\n        req=urllib.request.Request(self.url+'/api/predict',json.dumps(data).encode(),{'Content-Type':'application/json'})\n        try:\n            with urllib.request.urlopen(req) as r:return r.status,json.load(r)\n        except urllib.error.HTTPError as e:return e.code,json.load(e)\n    def test_prediction_matches_saved_model(self):\n        status,r=self.post(self.sample);self.assertEqual(status,200)\n        predictor=Predictor();p=float(predictor.model.predict_proba(pd.DataFrame([self.sample]))[0,1])\n        self.assertAlmostEqual(r['survival_probability'],p);self.assertEqual(r['survived'],int(p>=.5))\n    def test_schema_and_ui(self):\n        with urllib.request.urlopen(self.url+'/api/schema') as r: schema=json.load(r)\n        self.assertEqual({f['name'] for f in schema['fields']},set(self.sample))\n        with urllib.request.urlopen(self.url+'/') as r:self.assertIn('api/predict',r.read().decode())\n    def test_missing_extra_category_and_nonfinite(self):\n        for payload in [{},dict(self.sample,survived=1),dict(self.sample,gender='unknown'),dict(self.sample,age=float('nan')),dict(self.sample,age=True),dict(self.sample,age=-1),dict(self.sample,**{'class':1.5})]:\n            with self.subTest(payload=payload):self.assertEqual(self.post(payload)[0],422)\n    def test_nullable(self):\n        status,r=self.post(dict(self.sample,cabin_deck=None,emergency_response_time=None));self.assertEqual(status,200);self.assertTrue(0<=r['survival_probability']<=1)\n    def test_literal_none_preserved(self):\n        from train import load_data\n        import tempfile\n        with tempfile.NamedTemporaryFile(mode='w',suffix='.csv') as f:\n            f.write('medical_condition,survived\\nNone,0\\nAsthma,1\\n');f.flush();_,X,_=load_data(f.name)\n            self.assertEqual(X.medical_condition.iloc[0],'None')\nif __name__=='__main__':unittest.main()\n",encoding='utf-8')
print('생성:',p)


## 3. 학습 데이터 업로드
파일 선택 창에서 **6.csv**를 선택하세요. 이름이 달라도 CSV 한 개를 선택하면 됩니다.

In [ ]:
from google.colab import files
uploaded=files.upload()
csvs=[k for k in uploaded if k.lower().endswith('.csv')]
if len(csvs)!=1: raise ValueError('CSV 파일을 정확히 한 개 선택하세요.')
Path('data').mkdir(exist_ok=True)
Path('data/6.csv').write_bytes(uploaded[csvs[0]])
print('데이터 준비 완료')


## 4. 학습 및 정확도 평가
검증 데이터로 후보를 선택하고 테스트 정확도를 측정합니다. 85% 미만이어도 실제 결과를 그대로 표시합니다.

In [ ]:
subprocess.run([sys.executable,'train.py','--data','data/6.csv'],check=True)
metrics=json.loads(Path('artifacts/metrics.json').read_text())
print(f"테스트 정확도: {metrics['test']['accuracy']:.2%}")
print('85% 달성:',metrics['target_met'])
print('확률 데이터 진단:',metrics['probability_diagnostic'])


## 5. 입력·출력 확인과 API 테스트

In [ ]:
schema=json.loads(Path('artifacts/schema.json').read_text())
for f in schema['fields']: print(f['name'],f['type'],'결측 허용:',f['nullable'])
print('출력:',schema['output'])
subprocess.run([sys.executable,'-m','unittest','-v','test_api.py'],check=True)


## 6. API 서버 실행
이 셀을 재실행하면 이 노트북이 앞서 실행한 서버만 종료하고 다시 실행합니다.

In [ ]:
import time, urllib.request
if 'api_process' in globals() and api_process.poll() is None:
    api_process.terminate()
    api_process.wait(timeout=10)
server_log=open('server.log','w')
api_process=subprocess.Popen([sys.executable,'app.py','--port','8000'],stdout=server_log,stderr=subprocess.STDOUT)
for attempt in range(30):
    if api_process.poll() is not None: raise RuntimeError(Path('server.log').read_text())
    try:
        with urllib.request.urlopen('http://127.0.0.1:8000/api/health',timeout=2) as response: print(response.read().decode())
        break
    except Exception: time.sleep(.5)
else: raise RuntimeError('서버 실행 시간 초과: server.log 확인')


In [ ]:
sample=json.loads(Path('example_request.json').read_text())
request=urllib.request.Request('http://127.0.0.1:8000/api/predict',data=json.dumps(sample).encode(),headers={'Content-Type':'application/json'})
with urllib.request.urlopen(request) as response: print(json.loads(response.read()))


## 7. 웹페이지 표시
아래 화면에서 값을 입력하고 예측 버튼을 누릅니다. 런타임 종료 시 서버도 종료됩니다.
Colab의 브라우저/쿠키 환경 때문에 화면이 표시되지 않으면 ZIP의 README에 있는 로컬 실행 방법을 사용하세요.

In [ ]:
from google.colab import output
output.serve_kernel_port_as_iframe(8000,height=1100)


## 8. 실행 결과 다운로드 (선택)

In [ ]:
import shutil
archive=shutil.make_archive('/content/titanic_team6_result','zip',root_dir=project)
files.download(archive)


## 제출 자료
모델·API·웹페이지와 보고서는 이 저장소의 projects/titanic 폴더에 있습니다.
노트북 실행으로 GitHub PR을 자동 생성하지는 않습니다.

## 자료 출처
https://scikit-learn.org/stable/common_pitfalls.html
https://github.com/googlecolab/colabtools/blob/main/google/colab/output/_util.py
